<a href="https://colab.research.google.com/github/bendarbasil-svg/heart-disease-prediction/blob/main/heart_disease_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Предсказание болезни сердца

## О чём проект
Модель машинного обучения предсказывает наличие болезни сердца
у пациента по клиническим показателям (возраст, давление, холестерин,
результаты ЭКГ и др.).

## Данные
Датасет Heart Disease (Cleveland, UCI) — 303 пациента, 14 признаков.
Источник: https://archive.ics.uci.edu/dataset/45/heart+disease

## Задача
Бинарная классификация: 1 = есть болезнь, 0 = нет.

## Метрики
Основная метрика — **recall для класса «болен»**.
В медицине пропустить болезнь опаснее, чем дать ложную тревогу.

## Автор
[Василий Бендаржевский]

In [7]:
# ячейка 2 импорт

# Библиотеки для работы с данными
import pandas as pd
import numpy as np

# Для разделения данных и масштабирования
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Модели
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Метрики
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [2]:

# ячейка 3 загрузка
# Загружаем датасет Heart Disease
url = "https://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
df = pd.read_csv(url)

print("Размер данных:", df.shape)
df.head()

Размер данных: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0,normal,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,normal,0


In [3]:
# ячейка 4 проверка типов
# Размеры и типы
print("Размер:", df.shape)
print()
print("Типы данных:")
print(df.dtypes)
print()
print("Пропуски:")
print(df.isnull().sum())
print()
print("Первые 5 строк:")
print(df.head())

Размер: (303, 14)

Типы данных:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal         object
target        int64
dtype: object

Пропуски:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Первые 5 строк:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204 

## Проблема в столбце `thal`

В столбце `thal` смешаны два формата значений:
- Текстовые: `normal` (168), `reversible` (115), `fixed` (18)
- Цифровые: `1` (1 раз), `2` (1 раз)

Всего записей: 303. Цифровые значения (2 шт.) — аномалия,
их смысл неизвестен. Поэтому удалим эти 2 строки,
а оставшиеся текстовые значения закодируем через one-hot.

In [5]:
# Ячейка 6
# Убираем 2 строки с цифровыми значениями (1 и 2) — не знаем, что они значат
df = df[~df["thal"].isin(["1", "2"])]

# Кодируем оставшиеся значения через one-hot
df = pd.get_dummies(df, columns=["thal"], dtype=int)

print("Размер после чистки:", df.shape)
print("Столбцы:", df.columns.tolist())
print()
print(df.head())

Размер после чистки: (301, 16)
Столбцы: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'target', 'thal_fixed', 'thal_normal', 'thal_reversible']

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

   ca  target  thal_fixed  thal_normal  thal_reversible  
0   0       0           1            0                0  
1   3       1           0            1                0  
2   2       0           0            0                1  
3   0       0           0            1                0  
4   0       0      

In [10]:
# Ячейка 7 разделяемых на X и y
# X — все признаки, кроме цели
# y — цель (наличие болезни)
X = df.drop("target", axis=1)
y = df["target"]

print("Признаков:", X.shape[1])
print("Примеров:", X.shape[0])
print()
print("Баланс классов:")
print(y.value_counts())

Признаков: 15
Примеров: 301

Баланс классов:
target
0    218
1     83
Name: count, dtype: int64


In [12]:
# Ячейка 8
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)
print()
print("Баланс классов в train:")
print(y_train.value_counts())
print()
print("Баланс классов в test:")
print(y_test.value_counts())

Обучающая выборка: (240, 15)
Тестовая выборка: (61, 15)

Баланс классов в train:
target
0    174
1     66
Name: count, dtype: int64

Баланс классов в test:
target
0    44
1    17
Name: count, dtype: int64


In [13]:
#Ячейка 9 масштабирование
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("До масштабирования — первые 3 значения chol:")
print(X_train["chol"].values[:3])
print()
print("После масштабирования — те же 3 значения:")
print(X_train_scaled[:3, X_train.columns.get_loc("chol")])

До масштабирования — первые 3 значения chol:
[564 325 293]

После масштабирования — те же 3 значения:
[5.86807145 1.41437479 0.81806395]


In [14]:
#Ячейка 10 обучение
# Логистическая регрессия
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

# Предсказания
preds_lr = model_lr.predict(X_test_scaled)

print(f"Точность: {accuracy_score(y_test, preds_lr):.2%}")
print()
print("Матрица ошибок:")
print(confusion_matrix(y_test, preds_lr))
print()
print("Полный отчёт:")
print(classification_report(y_test, preds_lr, target_names=["Здоров", "Болен"]))

Точность: 83.61%

Матрица ошибок:
[[42  2]
 [ 8  9]]

Полный отчёт:
              precision    recall  f1-score   support

      Здоров       0.84      0.95      0.89        44
       Болен       0.82      0.53      0.64        17

    accuracy                           0.84        61
   macro avg       0.83      0.74      0.77        61
weighted avg       0.83      0.84      0.82        61



In [15]:
#Ячейка 11 коррекция обучения
model_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
model_balanced.fit(X_train_scaled, y_train)

preds_balanced = model_balanced.predict(X_test_scaled)

print(f"Точность: {accuracy_score(y_test, preds_balanced):.2%}")
print()
print("Матрица ошибок:")
print(confusion_matrix(y_test, preds_balanced))
print()
print("Отчёт:")
print(classification_report(y_test, preds_balanced, target_names=["Здоров", "Болен"]))

Точность: 86.89%

Матрица ошибок:
[[41  3]
 [ 5 12]]

Отчёт:
              precision    recall  f1-score   support

      Здоров       0.89      0.93      0.91        44
       Болен       0.80      0.71      0.75        17

    accuracy                           0.87        61
   macro avg       0.85      0.82      0.83        61
weighted avg       0.87      0.87      0.87        61



In [16]:
#Ячейка 12 меняем порог
probs = model_balanced.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (probs > threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    fn = cm[1][0]  # пропущено больных
    fp = cm[0][1]  # ложных тревог
    recall = 12 if fn == 0 else (17 - fn) / 17
    print(f"Порог {threshold}: пропущено больных = {fn}, ложных тревог = {fp}, recall = {recall:.2f}")

Порог 0.3: пропущено больных = 2, ложных тревог = 6, recall = 0.88
Порог 0.4: пропущено больных = 5, ложных тревог = 6, recall = 0.71
Порог 0.5: пропущено больных = 5, ложных тревог = 3, recall = 0.71
Порог 0.6: пропущено больных = 6, ложных тревог = 3, recall = 0.65
Порог 0.7: пропущено больных = 7, ложных тревог = 3, recall = 0.59


In [17]:
#Ячейка 13
# Финальный порог — выбираем 0.3 для максимизации recall больных
FINAL_THRESHOLD = 0.3

probs = model_balanced.predict_proba(X_test_scaled)[:, 1]
final_preds = (probs > FINAL_THRESHOLD).astype(int)

print(f"Финальная модель (порог {FINAL_THRESHOLD}):")
print()
print(f"Точность: {accuracy_score(y_test, final_preds):.2%}")
print()
print("Матрица ошибок:")
print(confusion_matrix(y_test, final_preds))
print()
print("Отчёт:")
print(classification_report(y_test, final_preds, target_names=["Здоров", "Болен"]))

Финальная модель (порог 0.3):

Точность: 86.89%

Матрица ошибок:
[[38  6]
 [ 2 15]]

Отчёт:
              precision    recall  f1-score   support

      Здоров       0.95      0.86      0.90        44
       Болен       0.71      0.88      0.79        17

    accuracy                           0.87        61
   macro avg       0.83      0.87      0.85        61
weighted avg       0.88      0.87      0.87        61



## Выводы

### Данные
- Датасет Heart Disease (Cleveland, UCI): 303 пациента, 14 признаков
- После чистки (удалены 2 строки с некорректным значением `thal`) — 301 запись
- Дисбаланс классов: 72% здоровых, 28% больных

### Подход
- Модель: логистическая регрессия с `class_weight="balanced"`
- Признаки масштабированы через `StandardScaler`
- `class_weight="balanced"` — критично для медицинской задачи с дисбалансом
- Порог классификации выбран на уровне **0.3** (а не стандартного 0.5) — для максимизации recall больных

### Результаты (тестовая выборка, 61 пациент)
- **Accuracy:** 86.89%
- **Recall для класса «болен»:** 0.88 — находим 15 из 17 больных
- **Precision для класса «болен»:** 0.71
- **Пропущено больных:** 2 (при стандартном пороге 0.5 — было бы 5)

### Что важнее accuracy
Основная метрика — **recall для класса «болен»**, потому что в медицине пропустить болезнь опаснее, чем дать ложную тревогу. Модель с accuracy 84% и recall 0.53 (обычная логистическая регрессия) **хуже** для клиники, чем модель с accuracy 87% и recall 0.88.

### Ограничения
- Тестовая выборка мала (61 пациент) — разница в 2–5 пропусках статистически шаткая
- Датасет 1988 года — на современных данных результаты могут отличаться
- Финальный порог требует согласования с врачом
- Модель — **вспомогательный инструмент**, а не замена врачебному решению

### Возможные улучшения
- Кросс-валидация для устойчивой оценки метрик
- ROC-кривая и AUC для выбора порога
- Сравнение с Random Forest и градиентным бустингом
- Проверка на внешнем датасете